# Best practice working

In [1]:
import torch
from safetensors import safe_open
import json
import psutil
import gc

# Function to check memory usage
def check_memory_usage():
    memory_info = psutil.virtual_memory()
    print(f"Memory Usage: {memory_info.percent}%, Available: {memory_info.available / (1024 ** 3):.2f} GB")

# Paths to the directory and files
model_dir = "./MODEL_FILES_V2"
model_weights_index_path = f"{model_dir}/model.safetensors.index.json"

# Function to incrementally convert safetensors to PyTorch format and save intermediate state dictionaries
def convert_safetensors_incremental(weights_index_path, model_dir, output_path):
    with open(weights_index_path, 'r') as f:
        index = json.load(f)
    
    weight_map = index['weight_map']
    state_dict = {}
    
    for i, (name, file) in enumerate(weight_map.items()):
        try:
            print(f"Loading {name} from {file}")
            with safe_open(f"{model_dir}/{file}", framework="pt") as f:
                tensor = f.get_tensor(name)
                state_dict[name] = tensor
                print(f"Loaded tensor {name} with shape {tensor.shape} and dtype {tensor.dtype}")
                check_memory_usage()
            
            # Save intermediate state dictionary every 10 tensors
            if (i + 1) % 10 == 0:
                intermediate_output_path = f"{output_path}_part_{(i + 1) // 10}.bin"
                torch.save(state_dict, intermediate_output_path)
                print(f"Saved intermediate state dictionary to {intermediate_output_path}")
                state_dict.clear()  # Clear the state dictionary to free memory
                gc.collect()  # Run garbage collection
        except Exception as e:
            print(f"Error loading {name} from {file}: {e}")
            break
    
    # Save the final state dictionary
    torch.save(state_dict, output_path)
    print(f"Final state dictionary saved to {output_path}")

# Convert and save the state dictionary incrementally
output_state_dict_path = f"{model_dir}/pytorch_model.bin"
convert_safetensors_incremental(model_weights_index_path, model_dir, output_state_dict_path)


Loading transformer.h.0.input_layernorm.bias from model-00001-of-00002.safetensors
Loaded tensor transformer.h.0.input_layernorm.bias with shape torch.Size([4544]) and dtype torch.float32
Memory Usage: 35.5%, Available: 10.09 GB
Loading transformer.h.0.input_layernorm.weight from model-00001-of-00002.safetensors
Loaded tensor transformer.h.0.input_layernorm.weight with shape torch.Size([4544]) and dtype torch.float32
Memory Usage: 35.5%, Available: 10.09 GB
Loading transformer.h.0.mlp.dense_4h_to_h.base_layer.weight from model-00001-of-00002.safetensors
Loaded tensor transformer.h.0.mlp.dense_4h_to_h.base_layer.weight with shape torch.Size([41295872, 1]) and dtype torch.uint8
Memory Usage: 35.5%, Available: 10.09 GB
Loading transformer.h.0.mlp.dense_4h_to_h.base_layer.weight.absmax from model-00001-of-00002.safetensors
Loaded tensor transformer.h.0.mlp.dense_4h_to_h.base_layer.weight.absmax with shape torch.Size([1290496]) and dtype torch.float32
Memory Usage: 35.5%, Available: 10.09 G

# Best practice loading the model!!? Inference?

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Paths to the directory and files
model_dir = "./MODEL_FILES_V2"
output_state_dict_path = f"{model_dir}/pytorch_model.bin"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# Load the model
model = AutoModelForCausalLM.from_pretrained(model_dir)

# Load the converted state dictionary
state_dict = torch.load(output_state_dict_path)
model.load_state_dict(state_dict, strict=False)
print("Model state dictionary loaded successfully.")

# Now you can use the model as usual
# Example: Generate some text (if it is a text generation model)
input_text = "Hello, how are you?"
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs)

# Decode the generated tokens
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated_text)


`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of the model checkpoint at ./MODEL_FILES_V2 were not used when initializing FalconForCausalLM: ['transformer.h.0.mlp.dense_4h_to_h.base_layer.weight', 'transformer.h.0.mlp.dense_4h_to_h.base_layer.weight.absmax', 'transformer.h.0.mlp.dense_4h_to_h.base_layer.weight.quant_map', 'transformer.h.0.mlp.dense_4h_to_h.base_layer.weight.quant_state.bitsandbytes__nf4', 'transformer.h.0.mlp.dense_4h_to_h.lora_A.default.weight', 'transformer.h.0.mlp.dense_4h_to_h.lora_B.default.weight', 'transformer.h.0.mlp.dense_h_to_4h.base_layer.weight', 'transformer.h.0.mlp.dense_h_to_4h.base_layer.weight.absmax', 'transformer.h.0.mlp.dense_h_to_4h.base_layer.weight.quant_map', 'transformer.h.0.mlp.dense_h_to_4h.base_layer.weight.quant_state.bitsandbytes__nf4', 'transformer.h.0.mlp.dense_h_to_4h.lora_A.default.weight', 'transformer.h.0.mlp.dense_h_to_4h.lora_B.default.weight', 'transformer.h.0.self_attention.dense.base_layer.weight', 'transformer.h.0.self_attention.dense.base_layer.weight.absmax'

OutOfMemoryError: CUDA out of memory. Tried to allocate 40.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 10.68 GiB is allocated by PyTorch, and 16.01 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# For testing

In [4]:
# Now you can use the model as usual
# Example: Generate some text (if it is a text generation model)
input_text = "Hello, how are you today?"
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs)

# Decode the generated tokens
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated_text)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
c:\Users\Marco\anaconda3\envs\Model_env\Lib\site-packages\transformers\generation\utils.py:1132: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
c:\Users\Marco\anaconda3\envs\Model_env\Lib\site-packages\transformers\generation\utils.py:1460: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Hello, how are you today? Reyn Sheldon Sheldon Romeo Urs Raven Raven Raven Raven Raven Raven Raven Raven
